# Reproduce supplementary Tables S3 and S5

This release notebook reconstructs Tables S3 and S5 for every model reported in the paper. It reads the evaluated metric JSON files from `output/metrics`, combines the two VISA subsets, preserves the camera-ready model order, and prints both pandas and LaTeX tables.

Run the inference and evaluation stages before executing this notebook.

## Expected metric outputs

Run `jobs/processing/metrics_processing.sh` before this notebook. The evaluation stage writes:

```text
output/metrics/
├── answers/
└── iou/        # precision and recall used for Tables S3 and S5
```

This notebook reads the `answerBox` and `box` metric JSON files from `output/metrics/iou`.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display


def find_repo_root(start=Path.cwd().resolve()):
    for candidate in (start, *start.parents):
        if (candidate / 'scripts').is_dir() and (candidate / 'README.md').is_file():
            return candidate
    raise FileNotFoundError('Run this notebook from inside the DocAttriBench repository.')


REPO_ROOT = find_repo_root()
METRICS_DIR = REPO_ROOT / 'output' / 'metrics'
ANSWER_DIR = METRICS_DIR / 'answers'
IOU_DIR = METRICS_DIR / 'iou'
THRESHOLD = 0.5
DATASETS = ['docvqa', 'visualmrc', 'visa', 'slidevqa', 'visualwebbench', 'longdocurl', 'mmlongbenchdoc']
VISA_PARTS = ('papervisa', 'wikivisa')

# Public output names plus legacy checkpoint names used for the camera-ready results.
PAPER_MODELS = {
    'InternVL2.5-2B': ('OpenGVLab_InternVL2_5-2B',),
    'InternVL3-2B': ('OpenGVLab_InternVL3-2B-Instruct',),
    'InternVL3.5-2B': ('OpenGVLab_InternVL3_5-2B-Instruct',),
    'Qwen2.5-VL-3B': ('Qwen_Qwen2.5-VL-3B-Instruct',),
    'Qwen3-VL-2B': ('Qwen_Qwen3-VL-2B-Instruct',),
    'InternVL2.5-8B': ('OpenGVLab_InternVL2_5-8B',),
    'InternVL3-8B': ('OpenGVLab_InternVL3-8B-Instruct',),
    'InternVL3.5-8B': ('OpenGVLab_InternVL3_5-8B-Instruct',),
    'Qwen2.5-VL-7B': ('Qwen_Qwen2.5-VL-7B-Instruct',),
    'Qwen3-VL-8B': ('Qwen_Qwen3-VL-8B-Instruct',),
    'InternVL2.5-38B': ('OpenGVLab_InternVL2_5-38B',),
    'InternVL3-38B': ('OpenGVLab_InternVL3-38B-Instruct',),
    'InternVL3.5-38B': ('OpenGVLab_InternVL3_5-38B-Instruct',),
    'Qwen2.5-VL-32B': ('Qwen_Qwen2.5-VL-32B-Instruct',),
    'Qwen3-VL-32B': ('Qwen_Qwen3-VL-32B-Instruct',),
    'VISA-7B': ('MrLight_visa-7B-single-fulldata-merged',),
    'MAPPET-7B': (
        'Mappet_Mappet-7B',
        'Mappet_Mappet2.5-VL-7B-Instruct-LoRA_v3_chkp_8400',
    ),
    'MAPPET-8B': (
        'Mappet_Mappet-8B',
        'Mappet_Mappet3-VL-8B-Instruct-LoRA_v3_chkp_11600',
    ),
    'MAPPET-I-8B': (
        'Mappet_Mappet-I-8B',
        'Mappet_MappetIntern3.5-VL-8B-Instruct-LoRA_v2_chkp_7600',
    ),
}
MODEL_ORDER = list(PAPER_MODELS)
MODEL_LABELS = {
    artifact: label
    for label, artifacts in PAPER_MODELS.items()
    for artifact in artifacts
}

In [ ]:
required_directories = {
    'bounding-box metrics': IOU_DIR,
}
missing_directories = [path for path in required_directories.values() if not path.is_dir()]
empty_directories = [
    path for path in required_directories.values()
    if path.is_dir() and not any(path.glob('*.json'))
]
if missing_directories or empty_directories:
    problems = [*(f'missing: {path}' for path in missing_directories),
                *(f'no JSON files: {path}' for path in empty_directories)]
    details = '\n'.join(f'  - {problem}' for problem in problems)
    raise FileNotFoundError(
        'Metric outputs are not ready. Run jobs/processing/metrics_processing.sh first.\n'
        f'{details}'
    )

print(f'Loading metric outputs from: {METRICS_DIR}')

In [ ]:
def threshold_metrics(path):
    with path.open() as handle:
        rows = json.load(handle)
    return next(row for row in rows if row['thr'] == THRESHOLD)


def load_task(suffixes):
    chosen = {}
    for path in sorted(IOU_DIR.glob('*.json')):
        if not path.name.endswith(suffixes):
            continue
        suffix = next(value[:-5] for value in suffixes if path.name.endswith(value))
        dataset, rest = path.stem.split('_', 1)
        model = rest[:-len(suffix)]
        metrics = threshold_metrics(path)
        key = (dataset, model)
        if key not in chosen or path.name.endswith('_clean.json'):
            chosen[key] = {
                'dataset': dataset,
                'model': model,
                'P_box': metrics['macro_p'] * 100,
                'R_box': metrics['macro_r'] * 100,
            }
    frame = pd.DataFrame(chosen.values())
    visa = frame[frame.dataset.isin(VISA_PARTS)].groupby('model', as_index=False)[['P_box', 'R_box']].mean()
    visa['dataset'] = 'visa'
    frame = pd.concat([frame[~frame.dataset.isin(VISA_PARTS)], visa], ignore_index=True)
    frame = frame[frame.model.isin(MODEL_LABELS)].copy()
    frame['Model'] = frame.model.map(MODEL_LABELS)
    return frame


def make_table(frame):
    table = frame.pivot(index='Model', columns='dataset', values=['P_box', 'R_box'])
    table = table.reindex(index=MODEL_ORDER, columns=pd.MultiIndex.from_product([['P_box', 'R_box'], DATASETS]))
    table[('Avg', 'P_box')] = table['P_box'].mean(axis=1)
    table[('Avg', 'R_box')] = table['R_box'].mean(axis=1)
    return table.round(1)


TASKS = {
    'Table S3: grounded answer generation': ('_answerBox.json', '_answerBox_clean.json'),
    'Table S5: answer locating': ('_box.json', '_box_clean.json'),
}

In [ ]:
tables = {}
for title, suffixes in TASKS.items():
    print(title)
    tables[title] = make_table(load_task(suffixes))
    display(tables[title])
    print(tables[title].to_latex(escape=False, na_rep='-'))
